# OOD Monitoring as a SOTIF Safety Measure

**Module 04 · SOTIF (ISO 21448)** · Notebook 3 of 5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/04_SOTIF_ISO21448/03_ood_monitoring_as_safety_measure.ipynb)

| Level | Prerequisites | Standards |
|---|---|---|
| Intermediate | [01 · SOTIF fundamentals](01_sotif_fundamentals.ipynb), [02/03 · OOD detection methods](../02_Failures_and_Robustness/03_out_of_distribution_detection.ipynb) | ISO 21448:2022, ISO/PAS 8800:2024, UN R157 |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

- Place an out-of-distribution (OOD) monitor correctly in the ISO 21448 causal chain and activity flow
- Extract runtime monitor signals from a **real detector** (scores, entropy, low-confidence boxes, energy, backbone
  features) and measure how well they reveal **degraded conditions** on real KITTI frames (AUROC)
- Set a monitor threshold **from a safety requirement** instead of an arbitrary percentile, and check it on held-out frames
- Convert a false-positive rate into **false alarms per hour**, state what a small clean sample can and cannot show,
  and apply a temporal filter to a real image sequence
- Treat the TPR/FPR trade-off as a **safety vs availability** decision and document it
- Choose a fallback that fits the **level of automation** (L2 warning, L3 transition demand, L4 minimal risk manoeuvre)
- State the limits: OOD is not the same as hazardous, and the monitor needs its own validation

This notebook does **not** re-derive OOD scoring methods; Mahalanobis, k-NN, MSP, energy scores and their
evaluation are covered in [02/03 · Out-of-distribution detection](../02_Failures_and_Robustness/03_out_of_distribution_detection.ipynb).
Here the question is how to turn a monitor signal into a **safety measure** that can be argued in a SOTIF case — and
the signals come from the same real detector and the same triggering conditions (fog, darkness) whose effect was
measured in [Notebook 02](02_triggering_conditions_and_scenarios.ipynb).

**Data and licence.** 30 KITTI object frames (image, LiDAR scan for the fog depth, labels, calibration) and 40
consecutive frames of KITTI tracking sequence 0015, fetched on demand by `avsafety.data` and cached; CC BY-NC-SA 3.0.

In [ ]:
# Course toolkit (tables, formulas and real-data loaders). Installs automatically on Colab.
try:
    import avsafety
except ImportError:
    %pip install -q "avsafety[data] @ git+https://github.com/milinpatel07/Autonomous-Driving_AI-Safety-and-Security.git"
    import avsafety
from avsafety import data as D

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage, stats
from sklearn.metrics import roc_auc_score, roc_curve
from IPython.display import display

rng = np.random.default_rng(42)
OI = {'orange': '#E69F00', 'sky': '#56B4E9', 'green': '#009E73', 'yellow': '#F0E442',
      'blue': '#0072B2', 'vermillion': '#D55E00', 'purple': '#CC79A7', 'grey': '#999999', 'black': '#000000'}
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})
print(D.attribution('KITTI'))

## 1. Where an OOD monitor sits in ISO 21448

A perception model trained on finite data has **performance insufficiencies** outside the conditions it was trained
and validated for. An OOD monitor estimates at runtime whether the current input resembles that data. In SOTIF terms:

| SOTIF concept | Role of the OOD monitor |
|---|---|
| Functional insufficiency (performance, training-data coverage) | The monitor flags inputs the model is unlikely to handle — a **functional modification** (clause 8) that mitigates the insufficiency |
| Triggering conditions | Logged OOD events are **candidate triggering conditions** for offline analysis (clause 7) |
| Area 3 (unknown, hazardous) | Field OOD logs support **discovery** of unknown scenarios (clauses 11 and 13) |
| Operation phase (clause 13) | Fleet-wide OOD statistics show drift of the operating environment |
| Hazardous behaviour | The monitor itself does nothing safe — only the **fallback** it triggers does |

Two limits must be stated in any safety argument:

- **OOD ≠ hazardous.** Many novel inputs are handled well (Area 4), and many failures happen on in-distribution
  inputs. The monitor reduces, not removes, the residual risk.
- **The monitor has insufficiencies of its own** (missed OOD, false alarms) and needs its own requirements and V&V.
  ISO/PAS 8800:2024 discusses such monitors as safety measures for AI elements (see [Module 06](../06_AI_Safety_ISO_PAS_8800/01_ai_safety_standards.ipynb)).

In [ ]:
def visualize_monitor_in_chain():
    """OOD monitor as a measure between perception output and vehicle behaviour."""
    fig, ax = plt.subplots(figsize=(14, 5))
    boxes = [('Sensors', 0.08, OI['sky']), ('Perception\nmodel', 0.25, OI['blue']), ('OOD\nmonitor', 0.42, OI['orange']),
             ('Decision\n(threshold,\ntemporal filter)', 0.59, OI['orange']), ('Fallback\nby SAE level', 0.76, OI['green']),
             ('Vehicle\nbehaviour', 0.92, OI['grey'])]
    for label, x, col in boxes:
        ax.add_patch(plt.Rectangle((x - 0.065, 0.42), 0.13, 0.3, facecolor=col, alpha=0.35, edgecolor='black'))
        ax.text(x, 0.57, label, ha='center', va='center', fontsize=10, fontweight='bold')
    for (_, x0, _), (_, x1, _) in zip(boxes[:-1], boxes[1:]):
        ax.annotate('', xy=(x1 - 0.067, 0.57), xytext=(x0 + 0.067, 0.57), arrowprops=dict(arrowstyle='->', lw=2))
    ax.annotate('', xy=(0.42, 0.40), xytext=(0.42, 0.20), arrowprops=dict(arrowstyle='<-', lw=1.5, ls='--'))
    ax.text(0.42, 0.13, 'OOD event log → candidate triggering conditions,\nArea 3 discovery, operation-phase monitoring',
            ha='center', fontsize=9, style='italic')
    ax.text(0.59, 0.86, 'safety requirement → threshold\nfalse alarms/h → availability', ha='center', fontsize=9)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
    ax.set_title('An OOD monitor as a SOTIF measure', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

visualize_monitor_in_chain()

## 2. Monitor signals from a real detector

The component is the camera detector of Notebook 02: `fasterrcnn_mobilenet_v3_large_fpn`, COCO-pretrained, on CPU.
Each frame yields six candidate monitor signals, all available at runtime without ground truth:

| Signal | Computed from | Idea |
|---|---|---|
| **Low top scores** | 1 − mean of the five highest detection scores | Maximum-softmax-probability baseline (Hendrycks & Gimpel, 2017) at detection level |
| **Proposal entropy** | Mean class entropy of the 50 most object-like region proposals | Uncertain classifications |
| **Low-confidence boxes** | Number of detections with 0.05 ≤ score < 0.5 | Objects seen but not confirmed |
| **Energy** | Mean $-\log\sum_c e^{f_c}$ over the same 50 proposals | Energy score (Liu et al., 2020) |
| **Feature norm** | Mean channel-L2 norm of the finest FPN feature map | Weak activations on low-contrast images |
| **Feature k-NN distance** | Distance of the pooled backbone feature vector to its nearest clean calibration frame | Deep nearest neighbours (Sun et al., 2022) |

Conditions are those of Notebook 02, regenerated here with the same models: Koschmieder fog from densified LiDAR
depth, and gain-corrected low light with shot and read noise. The ground-truth labels are used only to measure
recall — which decides whether a condition is **degraded** (pedestrian recall within the AEB range below the
derived requirement) or merely **shifted** (different from clear daylight, but still adequate).

In [ ]:
import torch
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn
torch.set_num_threads(min(4, os.cpu_count()))
detector = fasterrcnn_mobilenet_v3_large_fpn(weights="DEFAULT").eval()
_hook = {}
detector.backbone.register_forward_hook(lambda m, i, o: _hook.__setitem__('fpn', o))
detector.roi_heads.box_predictor.register_forward_hook(lambda m, i, o: _hook.__setitem__('logits', o[0]))

FRAMES = [518, 764, 961, 1357, 1693, 1988, 2160, 2379, 2595, 2775, 2966, 3438, 3560, 3763, 4235,
          4442, 4704, 4924, 5037, 5278, 5524, 5692, 5980, 6176, 6422, 6552, 6761, 6925, 7189, 7464]
AEB_RANGE_M, REQ_RECALL, SCORE_THR = 32.0, 0.80, 0.5          # as derived in Notebook 02
CONDITIONS = [('clear', None), ('fog', 150), ('fog', 50), ('fog', 25), ('low light', 1 / 64), ('low light', 1 / 256)]

def srgb_to_lin(x):
    return np.where(x <= 0.04045, x / 12.92, ((x + 0.055) / 1.055) ** 2.4)

def lin_to_srgb(x):
    x = np.clip(x, 0, 1)
    return np.where(x <= 0.0031308, 12.92 * x, 1.055 * x ** (1 / 2.4) - 0.055)

def dense_range(frame, cell=4):
    """Per-pixel distance from the projected LiDAR scan (as in Notebook 02, Section 5.1)."""
    H, W = frame.image.shape[:2]
    rect = frame.velo_to_rect(frame.points); rect = rect[rect[:, 2] > 1.0]
    uv, dist = frame.rect_to_image(rect), np.linalg.norm(rect, axis=1)
    ok = (uv[:, 0] >= 0) & (uv[:, 0] < W) & (uv[:, 1] >= 0) & (uv[:, 1] < H)
    gh, gw = -(-H // cell), -(-W // cell)
    grid = np.full((gh, gw), np.inf)
    np.minimum.at(grid, ((uv[ok, 1] // cell).astype(int), (uv[ok, 0] // cell).astype(int)), dist[ok])
    for c in range(gw):
        rows = np.flatnonzero(np.isfinite(grid[:, c]))
        if len(rows):
            grid[:rows[0], c] = grid[rows[0], c]
    idx = ndimage.distance_transform_edt(~np.isfinite(grid), return_distances=False, return_indices=True)
    return ndimage.zoom(ndimage.median_filter(grid[tuple(idx)], size=5), (H / gh, W / gw), order=1)[:H, :W]

def degrade(img, cond, p, dist=None, seed=0):
    if cond == 'fog':
        t = np.exp(np.log(0.05) / p * dist)[..., None]
        return (lin_to_srgb(srgb_to_lin(img / 255.0) * t + 0.8 * (1 - t)) * 255 + 0.5).astype(np.uint8)
    if cond == 'low light':
        r = np.random.default_rng(seed)
        e = r.poisson(srgb_to_lin(img / 255.0) * 4000.0 * p) + r.normal(0, 4.0, img.shape)
        return (lin_to_srgb(e / (4000.0 * p)) * 255 + 0.5).astype(np.uint8)
    return img

def run(img):
    """Detections plus the raw runtime signals of one frame."""
    x = torch.from_numpy(np.ascontiguousarray(img)).permute(2, 0, 1).float() / 255
    with torch.inference_mode():
        out = detector([x])[0]
    logits = _hook['logits']
    p = logits.softmax(1)
    top = torch.topk(1 - p[:, 0], k=min(50, len(p))).indices               # most object-like proposals
    ent = -(p[top] * p[top].clamp_min(1e-12).log()).sum(1) / np.log(p.shape[1])
    f0 = _hook['fpn']['0'][0]
    pooled = torch.cat([v[0].mean((1, 2)) for k, v in _hook['fpn'].items() if k != 'pool'])
    s = out['scores'].numpy()
    sig = {'low top scores': 1 - np.sort(s)[::-1][:5].mean() if len(s) else 1.0,
           'proposal entropy': float(ent.mean()),
           'low-confidence boxes': int(np.sum((s >= 0.05) & (s < SCORE_THR))),
           'energy': float(-torch.logsumexp(logits[top], 1).mean()),
           'feature norm': float(f0.norm(dim=0).mean())}
    return out['boxes'].numpy(), s, out['labels'].numpy(), sig, pooled.numpy()

def box_iou(a, b):
    lt = np.maximum(a[:, None, :2], b[None, :, :2]); rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.prod(np.clip(rb - lt, 0, None), axis=2)
    area = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter)

def n_detected(gt, boxes, scores):
    """Number of GT boxes matched (IoU ≥ 0.5, greedy by score) by detections above the operating threshold."""
    keep = scores >= SCORE_THR
    if not len(gt) or not keep.any():
        return 0
    M, hit = box_iou(gt, boxes[keep]), np.zeros(len(gt), bool)
    for j in np.argsort(-scores[keep]):
        cand = np.flatnonzero(~hit & (M[:, j] >= 0.5))
        if len(cand):
            hit[cand[np.argmax(M[cand, j])]] = True
    return int(hit.sum())

rows, feats = [], []
for i in FRAMES:
    f = D.kitti_object(i)
    dist = dense_range(f)
    peds = [l for l in f.labels if l['type'] == 'Pedestrian' and D.kitti_difficulty(l) in ('Easy', 'Moderate')
            and np.hypot(l['x'], l['z']) <= AEB_RANGE_M]
    gt = np.array([[l['x1'], l['y1'], l['x2'], l['y2']] for l in peds]).reshape(-1, 4)
    for cond, p in CONDITIONS:
        boxes, scores, labels, sig, pooled = run(degrade(f.image, cond, p, dist, seed=i))
        person = labels == 1
        rows.append({'frame': i, 'condition': cond if p is None else f'{cond} {p:.0f} m' if cond == 'fog' else f'{cond} 1/{1 / p:.0f}',
                     'n_ped': len(gt), 'n_hit': n_detected(gt, boxes[person], scores[person]), **sig})
        feats.append(pooled)
frames_df = pd.DataFrame(rows)
feats = np.array(feats)
print(f"{len(frames_df)} detector runs on {len(FRAMES)} frames × {len(CONDITIONS)} conditions.")

In [ ]:
cond_recall = frames_df.groupby('condition', sort=False)[['n_hit', 'n_ped']].sum()
cond_recall['recall'] = cond_recall['n_hit'] / cond_recall['n_ped']
cond_recall['class'] = np.where(cond_recall.index == 'clear', 'reference',
                                np.where(cond_recall['recall'] < REQ_RECALL, 'degraded', 'shifted, adequate'))
display(cond_recall.round(2))
frames_df['class'] = frames_df['condition'].map(cond_recall['class'])
DEGRADED = list(cond_recall.index[cond_recall['class'] == 'degraded'])
print(f"Degraded conditions (pedestrian recall within {AEB_RANGE_M:.0f} m below {REQ_RECALL:.0%}): {DEGRADED}")

## 3. Which signal reveals degraded conditions?

The frames are split into a **calibration** half (even positions in `FRAMES`) and a **test** half (odd positions):
anything fitted — the k-NN reference set, the orientation of a signal, later the threshold — uses calibration frames
only, and the numbers reported below come from test frames. Both halves are drawn from the same frame list, so they
share one distribution (an ID test set from a *different* distribution would make every frame look OOD).

AUROC is reported twice: degraded against clear frames, and degraded against clear **plus shifted-but-adequate**
frames. The second is the honest one for a safety monitor — an alarm on adequate frames costs availability without
buying safety.

In [ ]:
frames_df['split'] = np.where(frames_df['frame'].map({f: k for k, f in enumerate(FRAMES)}) % 2 == 0, 'cal', 'test')
cal_clear = (frames_df['split'] == 'cal') & (frames_df['class'] == 'reference')
ref = feats[cal_clear.to_numpy()]
d = np.linalg.norm(feats[:, None, :] - ref[None, :, :], axis=2)
same = frames_df['frame'].to_numpy()[:, None] == frames_df.loc[cal_clear, 'frame'].to_numpy()[None, :]
d[same] = np.inf                                     # a frame may not use its own clear version as neighbour
frames_df['feature k-NN distance'] = d.min(axis=1)

SIGNALS = ['low top scores', 'proposal entropy', 'low-confidence boxes', 'energy', 'feature norm', 'feature k-NN distance']
cal = frames_df[frames_df['split'] == 'cal']; test = frames_df[frames_df['split'] == 'test']
orient = {}
for s in SIGNALS:                                    # orientation chosen on calibration data only
    y = cal['class'] == 'degraded'
    orient[s] = 1 if roc_auc_score(y, cal[s]) >= 0.5 else -1

def auroc(df, s, negatives):
    sub = df[df['class'].isin(['degraded'] + negatives)]
    return roc_auc_score(sub['class'] == 'degraded', orient[s] * sub[s])

auc = pd.DataFrame({s: {'orientation': '+' if orient[s] > 0 else '−',
                        'AUROC degraded vs clear (test)': auroc(test, s, ['reference']),
                        'AUROC degraded vs clear + shifted (test)': auroc(test, s, ['reference', 'shifted, adequate'])}
                    for s in SIGNALS}).T
display(auc.style.format({'AUROC degraded vs clear (test)': '{:.2f}', 'AUROC degraded vs clear + shifted (test)': '{:.2f}'}))

cal_auc = {s: auroc(cal, s, ['reference', 'shifted, adequate']) for s in SIGNALS}
SIGNAL = max(cal_auc, key=cal_auc.get)
print(f"Signal selected on calibration data: '{SIGNAL}' (calibration AUROC {cal_auc[SIGNAL]:.2f}); "
      f"test AUROC against clear + shifted frames: {auc.loc[SIGNAL, 'AUROC degraded vs clear + shifted (test)']:.2f}. "
      f"Test set: {len(test)} frames, of which {int((test['class'] == 'degraded').sum())} degraded.")

In [ ]:
order = list(cond_recall.index)
colour = {'reference': OI['green'], 'shifted, adequate': OI['sky'], 'degraded': OI['vermillion']}
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
for k, c in enumerate(order):
    v = orient[SIGNAL] * frames_df.loc[frames_df['condition'] == c, SIGNAL]
    axes[0].scatter(np.full(len(v), k) + rng.uniform(-0.15, 0.15, len(v)), v, s=18,
                    color=colour[cond_recall.loc[c, 'class']], alpha=0.8)
axes[0].set_xticks(range(len(order)), order, rotation=20)
axes[0].set_ylabel(f"{'' if orient[SIGNAL] > 0 else '−'}{SIGNAL} (higher = more suspicious)")
axes[0].legend(handles=[plt.Line2D([], [], marker='o', ls='', color=v, label=k) for k, v in colour.items()], loc='upper left')
axes[0].set_title('Selected monitor signal per condition (all 30 frames)', fontweight='bold')
sub = test[test['class'].isin(['degraded', 'reference', 'shifted, adequate'])]
for s, col in zip(SIGNALS, [OI['blue'], OI['orange'], OI['green'], OI['purple'], OI['grey'], OI['vermillion']]):
    fpr, tpr, _ = roc_curve(sub['class'] == 'degraded', orient[s] * sub[s])
    axes[1].plot(fpr, tpr, color=col, lw=2, label=f"{s} ({auc.loc[s, 'AUROC degraded vs clear + shifted (test)']:.2f})")
axes[1].plot([0, 1], [0, 1], ':', color='k')
axes[1].set_xlabel('False-positive rate (clear + shifted test frames)'); axes[1].set_ylabel('True-positive rate (degraded test frames)')
axes[1].set_title('ROC on test frames (AUROC in brackets)', fontweight='bold'); axes[1].legend(fontsize=8, loc='lower right')
plt.tight_layout(); plt.show()

## 4. Setting the threshold from a safety requirement

A threshold such as "95th percentile of the clear-frame scores" fixes the **false-positive rate** and leaves the
detection rate to chance. A safety measure is specified the other way round, from what the hazard analysis needs.
An illustrative requirement:

> **R-MON-1** *The monitor shall flag at least 90 % of frames captured in degraded conditions (pedestrian recall
> within the AEB range below the derived requirement), demonstrated with 95 % confidence.*

Procedure: on the calibration frames, pick the **highest** threshold whose detection rate still has a one-sided
95 % Clopper–Pearson lower bound ≥ 0.90; then check detection rate and false-positive rate on the test frames.

**False alarms per hour.** At a frame rate $f$ a per-frame false-positive rate FPR gives
$\mathrm{FPR}\cdot f\cdot 3600$ flagged frames per hour before temporal filtering. With *zero* flagged frames among $N$
clean ones, the one-sided 95 % upper bound on FPR is about $3/N$ — so a few dozen clean frames cannot demonstrate a
low false-alarm rate. That needs hours of representative driving.

In [ ]:
FPS = 10                                     # KITTI camera frame rate
REQ_TPR, REQ_CONF = 0.90, 0.95

def cp_lower(k, n, conf=0.95):
    return 0.0 if k == 0 else stats.beta.ppf(1 - conf, k, n - k + 1)

def cp_upper(k, n, conf=0.95):
    return 1.0 if k == n else stats.beta.ppf(conf, k + 1, n - k)

score = lambda df: orient[SIGNAL] * df[SIGNAL].to_numpy()
pos_cal = score(cal[cal['class'] == 'degraded'])
t_req = None
for t in np.sort(np.unique(pos_cal))[::-1] - 1e-9:        # from strict to permissive
    if cp_lower(int((pos_cal > t).sum()), len(pos_cal), REQ_CONF) >= REQ_TPR:
        t_req = t; break
if t_req is None:
    t_req = np.quantile(pos_cal, 1 - REQ_TPR)
    print(f"R-MON-1 cannot be demonstrated with {len(pos_cal)} calibration frames — even flagging all of them gives a "
          f"lower bound of {cp_lower(len(pos_cal), len(pos_cal)):.3f}. Using the threshold with point estimate {REQ_TPR:.0%}.")
t_p95 = np.percentile(score(cal[cal['class'] == 'reference']), 95)

rows = []
for name, t in [('95th percentile of clear calibration frames', t_p95), ('From requirement R-MON-1', t_req)]:
    r = {'Threshold rule': name, 'threshold': t}
    for label, part in [('degraded', 'TPR'), ('reference', 'FPR clear'), ('shifted, adequate', 'FPR shifted')]:
        v = score(test[test['class'] == label]); k = int((v > t).sum())
        r[f'{part} (test)'] = f'{k}/{len(v)}'
        if part == 'TPR':
            r['TPR 95 % lower bound'] = cp_lower(k, len(v))
        if part == 'FPR clear':
            r['flagged clear frames/h (point; 95 % upper)'] = f'{k / len(v) * FPS * 3600:,.0f}; {cp_upper(k, len(v)) * FPS * 3600:,.0f}'
    rows.append(r)
display(pd.DataFrame(rows).round(3))
print(f"Text summary: the requirement-based threshold flags {rows[1]['TPR (test)']} degraded test frames, "
      f"{rows[1]['FPR clear (test)']} clear and {rows[1]['FPR shifted (test)']} shifted-but-adequate test frames. "
      f"The clear-frame sample bounds the false alarms only to ≤ {cp_upper(int((score(test[test['class'] == 'reference']) > t_req).sum()), int((test['class'] == 'reference').sum())) * FPS * 3600:,.0f} flagged frames per hour.")

## 5. A real sequence: tunnel entry and temporal filtering

Single frames are noisy evidence; a monitor decides on a **window**. An alarm is raised when at least $k$ of the last
$n$ frames are flagged. For independent frames with flag probability $p$ the window alarms with probability
$P(\mathrm{Bin}(n,p)\ge k)$, which falls steeply with $k$. Real consecutive frames are **correlated** — the same
scene, the same glare — so a short run of flagged frames fills the window and filtering helps less than the binomial
formula suggests.

The test below uses 40 consecutive frames (4 s at 10 fps) of KITTI tracking sequence 0015, a real urban drive. From
frame 62 to 77 the images are darkened (light factor 1/256 with gain and sensor noise), mimicking the entry into an
unlit underpass. The monitor uses the signal and threshold from above with a 3-of-5 filter.

In [ ]:
SEQ, SEQ_FRAMES, DARK = 15, list(range(50, 90)), range(62, 78)
K_F, N_F = 3, 5
seq = D.kitti_tracking(SEQ, frames=SEQ_FRAMES, images=True)
sig_seq = []
for fno, img in zip(SEQ_FRAMES, seq['images']):
    dark = fno in DARK
    _, _, _, sig, pooled = run(degrade(img, 'low light', 1 / 256, seed=fno) if dark else img)
    sig['feature k-NN distance'] = np.linalg.norm(ref - pooled, axis=1).min()
    sig_seq.append({'frame': fno, 'dark': dark, **sig})
sig_seq = pd.DataFrame(sig_seq)
flags = orient[SIGNAL] * sig_seq[SIGNAL].to_numpy() > t_req

def k_of_n(flags, k, n):
    win = np.convolve(flags.astype(int), np.ones(n, dtype=int), mode='full')[:len(flags)]
    return win >= k

alarm = k_of_n(flags, K_F, N_F)
t_s = (np.array(SEQ_FRAMES) - SEQ_FRAMES[0]) / FPS
fig, axes = plt.subplots(2, 1, figsize=(13, 5.5), sharex=True)
axes[0].plot(t_s, orient[SIGNAL] * sig_seq[SIGNAL], 'o-', color=OI['blue'], ms=4)
axes[0].axhline(t_req, color=OI['vermillion'], ls='--', label='threshold from R-MON-1')
axes[0].set_ylabel(SIGNAL); axes[0].legend(loc='upper left')
axes[1].step(t_s, flags.astype(int), where='post', color=OI['grey'], label='frame flagged')
axes[1].step(t_s, alarm.astype(int) * 0.9, where='post', color=OI['black'], lw=2, label=f'alarm ({K_F}-of-{N_F})')
axes[1].set_yticks([0, 1]); axes[1].set_xlabel('Time (s)'); axes[1].legend(loc='upper left')
for ax in axes:
    ax.axvspan((DARK.start - SEQ_FRAMES[0]) / FPS, (DARK.stop - SEQ_FRAMES[0]) / FPS, color=OI['vermillion'], alpha=0.12)
axes[0].set_title(f'KITTI tracking sequence {SEQ:04d}: darkness episode (shaded) and monitor response', fontweight='bold')
plt.tight_layout(); plt.show()

dark = sig_seq['dark'].to_numpy()
first = np.flatnonzero(alarm & dark)
false_onsets = int(np.sum(alarm[1:] & ~alarm[:-1] & ~dark[1:]) + (alarm[0] and not dark[0]))
print(f"Text summary: {int(flags[dark].sum())}/{dark.sum()} dark frames and {int(flags[~dark].sum())}/{(~dark).sum()} "
      f"normal frames flagged; alarm " + (f"raised {(first[0] - np.flatnonzero(dark)[0] + 1) / FPS:.1f} s after the episode starts"
      if len(first) else "not raised during the episode") + f"; {false_onsets} false alarm onsets in {(~dark).sum() / FPS:.1f} s of normal driving — "
      "far too little driving to estimate a rate per hour.")

## 6. Safety vs availability: the trade-off as a decision

Every threshold is a point on a curve: a higher detection rate of degraded frames (safety) costs more flagged
adequate frames (availability, and — for an automated vehicle — possibly new hazards from unnecessary manoeuvres).
The plot shows the curve on the test frames, separately for clear and for shifted-but-adequate frames.

> **Illustrative targets:** detection rate ≥ 90 % (R-MON-1) and a false-positive rate that, after filtering, gives
> at most one false alarm per ten hours are teaching values. Real targets come from the hazard analysis (cost of a
> missed degraded episode) and from the hazards and customer impact of the fallback.

In [ ]:
pos = score(test[test['class'] == 'degraded'])
neg_clear = score(test[test['class'] == 'reference']); neg_shift = score(test[test['class'] == 'shifted, adequate'])
ts = np.sort(np.r_[pos, neg_clear, neg_shift])[::-1]
curve = pd.DataFrame({'t': ts, 'TPR': [(pos > t).mean() for t in ts],
                      'FPR clear': [(neg_clear > t).mean() for t in ts], 'FPR shifted': [(neg_shift > t).mean() for t in ts]})
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(curve['FPR clear'], curve['TPR'], '-', color=OI['green'], lw=2, label='against clear test frames')
ax.plot(curve['FPR shifted'], curve['TPR'], '-', color=OI['sky'], lw=2, label='against shifted-but-adequate test frames')
ax.axhline(REQ_TPR, color=OI['vermillion'], ls='--', label=f'R-MON-1: TPR ≥ {REQ_TPR:.0%}')
ax.set_xlabel('Per-frame false-positive rate'); ax.set_ylabel('Per-frame detection rate (degraded)')
ax.set_title(f"Safety vs availability for '{SIGNAL}' (test frames)", fontweight='bold'); ax.legend(loc='lower right')
plt.tight_layout(); plt.show()
at_req = curve[curve['TPR'] >= REQ_TPR].iloc[0]
print(f"Text summary: reaching TPR ≥ {REQ_TPR:.0%} on test frames costs a per-frame FPR of {at_req['FPR clear']:.0%} on clear "
      f"and {at_req['FPR shifted']:.0%} on shifted-but-adequate frames, i.e. {at_req['FPR clear'] * FPS * 3600:,.0f} flagged clear frames "
      "per hour before filtering. Options to record in the SOTIF analysis: a better signal (combine signals, add a "
      "visibility or exposure sensor), a narrower ODD, a less disruptive first reaction, or a justified change of target.")

## 7. Fallback by level of automation

What an alarm should *do* depends on who is the fallback (SAE J3016:2021):

| Level | Who is the fallback | Fitting reaction to a sustained OOD alarm |
|---|---|---|
| **L2** (driver assistance) | The driver, who supervises continuously | Warn; request hands-on / eyes-on; degrade or switch off the function with a clear message |
| **L3** (conditional automation) | The fallback-ready user, after a **transition demand** | Issue a transition demand; if the driver does not take over, perform a minimal risk manoeuvre. For ALKS, UN Regulation No. 157 allows the MRM to start at the earliest 10 s after the transition demand begins (earlier for severe failures) |
| **L4** (high automation) | The ADS itself — no takeover is expected | Degrade (reduce speed, increase headway), then perform a **minimal risk manoeuvre** to reach a minimal risk condition |

A false alarm is not free: an unnecessary minimal risk manoeuvre on a motorway is itself a hazardous behaviour and
belongs in the hazard analysis.

In [ ]:
def fallback_action(level, state):
    """Reaction to the monitor state ('normal', 'caution', 'alarm') for SAE level 2, 3 or 4."""
    if state == 'normal':
        return 'Continue'
    if level == 2:
        return 'Driver warning' if state == 'caution' else 'Hands-on request; function degrades with message'
    if level == 3:
        return 'Increase headway, prepare' if state == 'caution' else 'Transition demand; MRM if no takeover'
    if level == 4:
        return 'Reduce speed, increase headway' if state == 'caution' else 'Minimal risk manoeuvre'
    raise ValueError("level must be 2, 3 or 4")


class RuntimeMonitor:
    """Requirement-based threshold + k-of-n temporal filter + level-dependent fallback."""

    def __init__(self, threshold, k, n, level, caution=2):
        self.threshold, self.k, self.n, self.level, self.caution = threshold, k, n, level, caution
        self.history = []

    def process(self, signal_value):
        self.history = (self.history + [signal_value > self.threshold])[-self.n:]
        flagged = sum(self.history)
        state = 'alarm' if flagged >= self.k else ('caution' if flagged >= self.caution else 'normal')
        return {'flagged_in_window': flagged, 'state': state, 'action': fallback_action(self.level, state)}

values = orient[SIGNAL] * sig_seq[SIGNAL].to_numpy()
for level in (2, 3, 4):
    mon = RuntimeMonitor(t_req, K_F, N_F, level)
    out = pd.DataFrame([mon.process(v) for v in values])
    acts = out.loc[sig_seq['dark'].to_numpy(), 'action'].value_counts()
    print(f"L{level} reactions during the darkness episode: " + "; ".join(f"{a} ({c} frames)" for a, c in acts.items()))

## 8. What goes into the SOTIF argument

- **Requirement and rationale:** which OOD conditions the monitor must detect, with which rate and latency, derived
  from the hazard analysis.
- **Evidence:** detection rate with a confidence bound on held-out frames that represent the degraded
  conditions; false-alarm rate per hour on representative, *temporally correlated* clean driving — hours of it, not
  the few dozen frames used here.
- **Fallback:** appropriate to the level of automation; hazards of the fallback analysed.
- **Limitations:** degraded inputs the monitor does not separate (here, the overlap with shifted-but-adequate frames),
  conditions it was never calibrated on (rain, glare), and failures on clear frames it cannot see — these remain
  residual risk to be bounded by validation (clauses 10–11).
- **Operation phase:** OOD events are logged and analysed; confirmed new triggering conditions update the analysis.

Uncertainty estimates of the perception model itself can serve a similar purpose; see the author's work on
uncertainty evaluation for identifying performance insufficiencies (Patel & Jung, 2026) and
[Module 06](../06_AI_Safety_ISO_PAS_8800/02_uncertainty_types.ipynb).

## 9. Exercise

1. Combine two signals: standardise them on the clear calibration frames and use their sum. Does the test AUROC against
   clear + shifted frames improve over the best single signal?
2. Re-set the threshold with `REQ_TPR = 0.95`. How many calibration frames would you need to demonstrate it if the
   monitor flagged every degraded frame?
3. Change the filter to 5-of-10. What happens to the alarm latency in the tunnel sequence, and why is latency itself
   a safety requirement (compare it with the FTTI of SG-02, 500 ms)?

In [ ]:
# Exercise starter — runs as is; edit the two signal names
A, B = 'proposal entropy', 'feature k-NN distance'
ref_clear = cal[cal['class'] == 'reference']
z = lambda df, s: orient[s] * (df[s] - ref_clear[s].mean()) / ref_clear[s].std()
combo_test = z(test, A) + z(test, B)
sub = test['class'].isin(['degraded', 'reference', 'shifted, adequate'])
print(f"Combined signal '{A}' + '{B}': test AUROC (degraded vs clear + shifted) = "
      f"{roc_auc_score(test.loc[sub, 'class'] == 'degraded', combo_test[sub]):.2f}; "
      f"best single signal '{SIGNAL}': {auc.loc[SIGNAL, 'AUROC degraded vs clear + shifted (test)']:.2f}")
print(f"Frames needed to demonstrate TPR ≥ 0.95 with zero misses: {int(np.ceil(np.log(0.05) / np.log(0.95)))}")

## In industry

**Work product.** The monitor appears in three places: as a **functional modification** in the SOTIF analysis
(clause 8) with its own requirements (detection rate, latency, false-alarm rate); as **ODD monitoring** in the
function specification (which conditions trigger which degradation); and as a **safety performance indicator** in
operation (clause 13), where flagged episodes are logged, triaged and fed back into the triggering-condition
analysis. ISO/PAS 8800 treats such monitors as safety mechanisms for AI elements.

**Who produces it.** Perception engineers implement the signals; the safety engineer derives R-MON-type requirements
from the HARA and the triggering-condition analysis; the function owner and HMI team define the fallback per
automation level; the field-monitoring team owns the operation-phase data loop.

**What reviewers and assessors check.**

- [ ] Monitor requirement traced to a hazardous behaviour and a measured performance insufficiency, not to "OOD" in general
- [ ] Calibration and evaluation on disjoint data; the conditions used match the triggering-condition analysis
- [ ] Detection rate with confidence bound; false alarms per hour measured on hours of representative, correlated driving
- [ ] Latency of the filtered alarm compared with the fault-tolerant time interval of the affected safety goal
- [ ] Fallback appropriate to the automation level, and its own hazards analysed
- [ ] Known blind spots stated (conditions never calibrated, failures on in-distribution inputs)

## Key takeaways

- An OOD monitor is a **functional modification** against performance insufficiencies, and its logs support
  discovery of triggering conditions and Area 3 scenarios. It is not a detector of hazardous scenarios.
- Monitor signals come from the **released detector** and are evaluated on the **triggering conditions** whose
  effect was measured; the label is *degraded* (performance below requirement), not merely *different*.
- Derive the threshold from a **safety requirement** with a confidence bound; then compute the cost in
  **false alarms per hour** — percentiles of training scores fix the wrong quantity.
- A few dozen clean frames cannot demonstrate a low false-alarm rate (≈ 3/N upper bound); temporal filtering trades
  latency for availability and is less effective on correlated real frames than on independent ones.
- When both safety and availability targets cannot be met, the gap is a **documented design decision**: better signal,
  narrower ODD or different fallback.
- The fallback depends on the level of automation: warning (L2), transition demand then MRM (L3), MRM (L4).

## Self-assessment

1. Why is "flags OOD inputs" not the same as "detects unknown hazardous scenarios"?
2. A monitor has a per-frame FPR of 1 % at 20 fps. How many frames per hour are flagged on ID data?
3. Why should the threshold be set from the detection requirement rather than from the 95th percentile of
   clear-frame scores?
4. Why does temporal filtering work worse on real driving data than on independent samples?
5. What should an L4 vehicle do on a sustained OOD alarm, and why is "request driver takeover" wrong for it?
6. Why is AUROC against clear *and shifted-but-adequate* frames the more relevant number for a safety monitor?
7. A monitor flags 0 of 15 clear test frames. What can you claim about false alarms per hour at 10 fps?

<details>
<summary>Answers</summary>

1. OOD measures novelty relative to training data. Novel inputs can be handled safely (Area 4), and hazardous
   behaviour can occur on in-distribution inputs; the monitor covers neither case.
2. 0.01 × 20 × 3600 = 720 flagged frames per hour.
3. The safety argument needs a guaranteed detection rate for specified degraded conditions; a clear-score percentile
   fixes only the false-positive rate and leaves the detection rate unspecified.
4. Consecutive frames are correlated, so ID frames that exceed the threshold come in runs; a k-of-n window is then
   often filled by a single run.
5. Degrade and perform a minimal risk manoeuvre. At Level 4 no user is expected to take over; the ADS is its own fallback.
6. Alarms on shifted frames where the function still meets its requirement cost availability (and may create new
   hazards through unnecessary manoeuvres) without reducing risk; a monitor must separate *degraded* from *different*.
7. Only an upper bound: with 0 of 15, the one-sided 95 % bound on FPR is 1 − 0.05^(1/15) ≈ 18 %, i.e. up to about
   6,600 flagged frames per hour before filtering. Demonstrating a low rate needs hours of representative driving.
</details>

## References

- ISO 21448:2022 *Road vehicles — Safety of the intended functionality*.
- ISO/PAS 8800:2024 *Road vehicles — Safety and artificial intelligence*. https://www.iso.org/standard/83303.html
- SAE J3016 (2021) *Taxonomy and Definitions for Terms Related to Driving Automation Systems for On-Road Motor Vehicles*.
- UN Regulation No. 157 — Automated Lane Keeping Systems (ALKS); EU publication: https://eur-lex.europa.eu/eli/reg/2021/389/oj
- Henriksson, J., Ursing, S., Erdogan, M., Warg, F., Thorsén, A., Jaxing, J., Örsmark, O., Örtenberg Toftås, M. (2023). Out-of-Distribution Detection as Support for Autonomous Driving Safety Lifecycle. *REFSQ 2023*, LNCS 13975, 233–242. https://doi.org/10.1007/978-3-031-29786-1_16
- Hendrycks, D., Gimpel, K. (2017). A Baseline for Detecting Misclassified and Out-of-Distribution Examples in Neural Networks. *ICLR 2017*. arXiv:1610.02136
- Liu, W., Wang, X., Owens, J., Li, Y. (2020). Energy-based Out-of-distribution Detection. *NeurIPS 2020*. arXiv:2010.03759
- Geiger, A., Lenz, P., Stiller, C., Urtasun, R. (2013). Vision meets robotics: The KITTI dataset. *International Journal of Robotics Research* 32(11), 1231–1237. https://doi.org/10.1177/0278364913491297
- Sakaridis, C., Dai, D., Van Gool, L. (2018). Semantic Foggy Scene Understanding with Synthetic Data. *International Journal of Computer Vision* 126, 973–992. https://doi.org/10.1007/s11263-018-1072-8
- Howard, A., et al. (2019). Searching for MobileNetV3. *IEEE/CVF ICCV 2019*, 1314–1324. https://doi.org/10.1109/ICCV.2019.00140
- Lee, K., Lee, K., Lee, H., Shin, J. (2018). A Simple Unified Framework for Detecting Out-of-Distribution Samples and Adversarial Attacks. *NeurIPS 2018*. arXiv:1807.03888
- Sun, Y., Ming, Y., Zhu, X., Li, Y. (2022). Out-of-Distribution Detection with Deep Nearest Neighbors. *ICML 2022*. arXiv:2204.06507
- Patel, M., Jung, R. (2026). Uncertainty Evaluation to Support Safety of the Intended Functionality Analysis for Identifying Performance Insufficiencies in ML-Based LiDAR Object Detection. *VEHITS 2026*, 154–167. https://doi.org/10.5220/0014967200004030
- Patel, M., Jung, R. (2026). Uncertainty Representation in a SOTIF-Related Use Case with Dempster-Shafer Theory for LiDAR Sensor-Based Object Detection. *CCIS*, Springer, 178–193. https://doi.org/10.1007/978-3-032-23187-1_10

## Further practice

For exercises on diagnosis methods and safety envelopes, see the author's companion repository
*AI Perception Safety Workshop* (external): https://github.com/milinpatel07/AI-Perception-Safety-Workshop

---
**Navigation:** ← [prev: Triggering conditions and scenarios](02_triggering_conditions_and_scenarios.ipynb) · [Module index](README.md) · [next: Simulation-based validation](04_simulation_based_validation.ipynb) →